# 01 · Preparação e qualidade dos dados — Nascidos vivos (SINASC/CE)

**Objetivo:** ler o CSV bruto, avaliar a qualidade (faltantes, duplicidades, período), converter as variáveis para o tipo correto,
tratar os códigos de "ignorado" e salvar uma base tratada que será usada nos notebooks 02 e 03.

> Execute este notebook **primeiro**. Ele gera o arquivo `nascidos_vivos_ce_tratado.csv` na mesma pasta do CSV bruto.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


CANDIDATOS = [
    Path("../../data/processed/nascidos_vivos_ce.csv"),
]
CAMINHO = next((p for p in CANDIDATOS if p.exists()), None)
if CAMINHO is None:
    raise FileNotFoundError("nascidos_vivos_ce.csv não encontrado. Ajuste a lista CANDIDATOS.")
print("Arquivo:", CAMINHO.resolve())

Arquivo: C:\Users\italo\OneDrive\Ambiente de Trabalho\Sinasc\Nascidos_Vivos\data\processed\nascidos_vivos_ce.csv


## 1. Leitura do arquivo bruto
Lemos tudo como texto (`dtype=str`) para **não perder zeros à esquerda** em códigos (ex.: `TPROBSON = "08"`) e converter só o que for numérico de fato.

In [6]:
df = pd.read_csv(CAMINHO, dtype=str)
print("Linhas x colunas:", df.shape)
df.head()

Linhas x colunas: (142568, 62)


,ARQUIVO_ORIGEM,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,QTDFILMORT,CODMUNRES,GESTACAO,GRAVIDEZ,PARTO,CONSULTAS,DTNASC,HORANASC,SEXO,APGAR1,APGAR5,RACACOR,PESO,IDANOMAL,DTCADASTRO,CODANOMAL,NUMEROLOTE,VERSAOSIST,DTRECEBIM,DIFDATA,DTRECORIGA,NATURALMAE,CODMUNNATU,CODUFNATU,ESCMAE2010,SERIESCMAE,DTNASCMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,IDADEPAI,DTULTMENST,SEMAGESTAC,TPMETESTIM,CONSPRENAT,MESPRENAT,TPAPRESENT,STTRABPART,STCESPARTO,TPNASCASSI,TPFUNCRESP,TPDOCRESP,DTDECLARAC,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK,CONTADOR
0,DNCE2025,1,6205224,110020,1,42,2,5,848310,NaN,NaN,230730,4,2,2,4,04052025,0255,2,08,08,1,0995,2,21052025,NaN,20250008,3.2.50,30052025,026,NaN,823,230270,23,5,NaN,22051982,1,NaN,NaN,NaN,NaN,NaN,32,9,07,02,1,2,2,1,2,3,06052025,08,1,1,1,08,0,5,7185
1,DNCE2025,1,6205224,110020,1,42,2,5,223505,NaN,NaN,230730,4,2,2,4,04052025,0258,2,07,07,1,1630,2,21052025,NaN,20250008,3.2.50,30052025,026,NaN,823,230270,23,5,NaN,22051982,1,NaN,NaN,NaN,NaN,NaN,32,9,07,02,1,2,2,1,2,3,06052025,08,1,1,1,08,0,5,7186
2,DNCE2025,1,9917322,150140,1,32,1,4,411010,01,00,230930,4,1,2,4,09022025,1159,1,08,09,4,2914,2,21022025,NaN,20250008,3.2.50,06032025,025,NaN,815,150080,15,3,NaN,19031992,4,01,00,01,NaN,28052024,36,8,12,03,1,2,2,1,5,4,09022025,12,1,1,1,10,1,5,69445
3,DNCE2025,1,2329905,150680,1,21,1,4,999992,00,00,230920,5,1,1,4,23062025,0447,2,08,09,4,3390,2,26062025,NaN,20250061,3.2.50,27062025,004,NaN,815,150060,15,3,1,12032004,4,00,00,00,NaN,23092024,39,8,08,02,1,9,3,2,2,3,23062025,05,1,1,1,01,0,5,110566
4,DNCE2025,1,2463016,210005,1,25,1,4,999992,02,01,231000,5,NaN,2,3,05052025,1510,2,08,09,4,4545,2,08052025,NaN,20250016,3.2.50,16052025,011,NaN,821,210005,21,3,3,21032000,4,03,01,02,NaN,15072024,41,8,05,05,1,2,2,1,5,4,05052025,06,1,1,1,11,1,2,134831


## 2. Qualidade dos dados
### 2.1 Valores faltantes (NaN) por coluna

In [7]:
faltantes = (
    df.isna().sum().to_frame("n_faltantes")
    .assign(pct=lambda d: (d["n_faltantes"] / len(df) * 100).round(2))
    .sort_values("pct", ascending=False)
)
display(faltantes[faltantes["n_faltantes"] > 0])

,n_faltantes,pct
DTRECORIGA,142568,100.00
CODANOMAL,140876,98.81
IDADEPAI,130043,91.21
SERIESCMAE,63885,44.81
DTULTMENST,62458,43.81
QTDFILMORT,20088,14.09
QTDPARTNOR,19416,13.62
QTDPARTCES,18758,13.16
RACACORMAE,14813,10.39
RACACOR,13994,9.82


### 2.2 Duplicidades e período coberto

In [8]:
data = pd.to_datetime(df["DTNASC"], format="%d%m%Y", errors="coerce")

print("Registros:", len(df))
print("CONTADOR único dentro de cada arquivo de origem?", not df.duplicated(["ARQUIVO_ORIGEM", "CONTADOR"]).any())
print("CONTADOR repetido entre arquivos (2025 x 2026):", int(df["CONTADOR"].duplicated().sum()), "-> não é erro; o contador reinicia em cada arquivo")
print("Linhas totalmente duplicadas:", int(df.duplicated().sum()))
print("Datas de nascimento inválidas:", int(data.isna().sum()))
print("Período:", data.min().date(), "a", data.max().date())
print()
print("Registros por arquivo de origem:")
print(df["ARQUIVO_ORIGEM"].value_counts())

Registros: 142568
CONTADOR único dentro de cada arquivo de origem? True
CONTADOR repetido entre arquivos (2025 x 2026): 19 -> não é erro; o contador reinicia em cada arquivo
Linhas totalmente duplicadas: 0
Datas de nascimento inválidas: 0
Período: 2025-01-01 a 2026-05-29

Registros por arquivo de origem:
ARQUIVO_ORIGEM
DNCE2025    107346
DNCE2026     35222
Name: count, dtype: int64


> **Atenção ao interpretar:** o arquivo de 2025 cobre o ano inteiro, mas o de 2026 cobre apenas **parte do ano** (até o fim de maio).
> Por isso, comparações entre anos devem ser feitas com cuidado (use proporções, não contagens).

## 3. Conversão das variáveis numéricas e tratamento de "ignorado"
No SINASC, o código **99** (e **9999** para peso) significa *ignorado*. Esses valores **não são medidas reais** e distorceriam média,
desvio padrão e quartis (ex.: um Apgar "99"), então são convertidos para `NaN`.

Valores extremos que *parecem* errados, mas não são códigos de ignorado (ex.: peso de 115 g), **não são removidos** aqui — eles serão tratados
como *outliers* e discutidos no notebook 02.

In [9]:
NUMERICAS = {
    "IDADEMAE": "Idade da mãe (anos)",
    "PESO": "Peso ao nascer (g)",
    "SEMAGESTAC": "Semanas de gestação",
    "CONSPRENAT": "Consultas de pré-natal",
    "APGAR1": "Apgar 1º minuto",
    "APGAR5": "Apgar 5º minuto",
    "QTDFILVIVO": "Filhos vivos anteriores",
    "QTDFILMORT": "Filhos mortos anteriores",
    "QTDGESTANT": "Gestações anteriores",
    "QTDPARTNOR": "Partos vaginais anteriores",
    "QTDPARTCES": "Partos cesáreos anteriores",
    "IDADEPAI": "Idade do pai (anos)",
}
CODIGO_IGNORADO = {"PESO": 9999}  # para as demais, o código é 99

tratado = pd.DataFrame({
    "CONTADOR": df["CONTADOR"],
    "ARQUIVO_ORIGEM": df["ARQUIVO_ORIGEM"],
    "CODMUNNASC": df["CODMUNNASC"],
    "CODMUNRES": df["CODMUNRES"],
})

resumo_ignorados = []
for col in NUMERICAS:
    bruto = pd.to_numeric(df[col], errors="coerce")
    cod = CODIGO_IGNORADO.get(col, 99)
    n_ign = int((bruto == cod).sum())
    tratado[col] = bruto.mask(bruto == cod)
    resumo_ignorados.append({
        "variavel": col,
        "codigo_ignorado": cod,
        "n_ignorados_convertidos": n_ign,
        "n_faltantes_final": int(tratado[col].isna().sum()),
    })

display(pd.DataFrame(resumo_ignorados))

,variavel,codigo_ignorado,n_ignorados_convertidos,n_faltantes_final
0,IDADEMAE,99,1,1
1,PESO,9999,0,4
2,SEMAGESTAC,99,0,4263
3,CONSPRENAT,99,346,3530
4,APGAR1,99,17,534
5,APGAR5,99,14,552
6,QTDFILVIVO,99,0,13221
7,QTDFILMORT,99,0,20088
8,QTDGESTANT,99,0,11835
9,QTDPARTNOR,99,0,19416


## 4. Variáveis derivadas e rótulos das categorias

In [10]:
tratado["DTNASC"] = data
tratado["ANO"] = data.dt.year
tratado["MES"] = data.dt.month


def rotular(coluna, mapa, rotulo_nd="Não informado"):
    codigos = pd.to_numeric(df[coluna], errors="coerce")
    return codigos.map(mapa).fillna(rotulo_nd)


tratado["SEXO_DESC"] = rotular("SEXO", {1: "Masculino", 2: "Feminino", 0: "Ignorado", 9: "Ignorado"})
tratado["PARTO_DESC"] = rotular("PARTO", {1: "Vaginal", 2: "Cesáreo", 9: "Ignorado"})
tratado["GRAVIDEZ_DESC"] = rotular("GRAVIDEZ", {1: "Única", 2: "Dupla", 3: "Tripla ou mais", 9: "Ignorado"})
tratado["GESTACAO_DESC"] = rotular("GESTACAO", {
    1: "< 22 sem", 2: "22–27 sem", 3: "28–31 sem", 4: "32–36 sem", 5: "37–41 sem", 6: "≥ 42 sem", 9: "Ignorado"})
tratado["RACACOR_DESC"] = rotular("RACACOR", {1: "Branca", 2: "Preta", 3: "Amarela", 4: "Parda", 5: "Indígena"})
tratado["ESCMAE_DESC"] = rotular("ESCMAE", {
    1: "Nenhuma", 2: "1 a 3 anos", 3: "4 a 7 anos", 4: "8 a 11 anos", 5: "12 anos ou mais", 9: "Ignorado"})

# Indicadores binários (1 = sim, 0 = não, NaN = sem informação). A média vira a proporção.
peso, sem = tratado["PESO"], tratado["SEMAGESTAC"]
parto = pd.to_numeric(df["PARTO"], errors="coerce")
tratado["BAIXO_PESO"] = (peso < 2500).astype(float).where(peso.notna())      # < 2.500 g
tratado["PREMATURO"] = (sem < 37).astype(float).where(sem.notna())           # < 37 semanas
tratado["CESAREO"] = (parto == 2).astype(float).where(parto.isin([1, 2]))

tratado.head()

,CONTADOR,ARQUIVO_ORIGEM,CODMUNNASC,CODMUNRES,IDADEMAE,PESO,SEMAGESTAC,CONSPRENAT,APGAR1,APGAR5,QTDFILVIVO,QTDFILMORT,QTDGESTANT,QTDPARTNOR,QTDPARTCES,IDADEPAI,DTNASC,ANO,MES,SEXO_DESC,PARTO_DESC,GRAVIDEZ_DESC,GESTACAO_DESC,RACACOR_DESC,ESCMAE_DESC,BAIXO_PESO,PREMATURO,CESAREO
0,7185,DNCE2025,110020,230730,42.0,995.0,32.0,7.0,8.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-04,2025,5,Feminino,Cesáreo,Dupla,32–36 sem,Branca,12 anos ou mais,1.0,1.0,1.0
1,7186,DNCE2025,110020,230730,42.0,1630.0,32.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-04,2025,5,Feminino,Cesáreo,Dupla,32–36 sem,Branca,12 anos ou mais,1.0,1.0,1.0
2,69445,DNCE2025,150140,230930,32.0,2914.0,36.0,12.0,8.0,9.0,1.0,0.0,1.0,0.0,1.0,NaN,2025-02-09,2025,2,Masculino,Cesáreo,Única,32–36 sem,Parda,8 a 11 anos,0.0,1.0,1.0
3,110566,DNCE2025,150680,230920,21.0,3390.0,39.0,8.0,8.0,9.0,0.0,0.0,0.0,0.0,0.0,NaN,2025-06-23,2025,6,Feminino,Vaginal,Única,37–41 sem,Parda,8 a 11 anos,0.0,0.0,0.0
4,134831,DNCE2025,210005,231000,25.0,4545.0,41.0,5.0,8.0,9.0,2.0,1.0,3.0,1.0,2.0,NaN,2025-05-05,2025,5,Feminino,Cesáreo,Não informado,37–41 sem,Parda,8 a 11 anos,0.0,0.0,1.0


## 5. Salvando a base tratada

In [ ]:
SAIDA = CAMINHO.parent / "SINASC_CE_PADRÃO_OURO_tratado.csv"
tratado.to_csv(SAIDA, index=False)
print("Base tratada salva em:", SAIDA.resolve())
print("Formato:", tratado.shape)

Base tratada salva em: C:\Users\italo\OneDrive\Ambiente de Trabalho\Sinasc\Nascidos_Vivos\data\processed\nascidos_vivos_ce_tratado.csv
Formato: (142568, 28)
